In [ ]:
#disclaimer, generated by GEN AI

import pandas as pd
import torch
import difflib
from torch.utils.data import Dataset, DataLoader
from transformers import DistilBertTokenizerFast
from sklearn.model_selection import train_test_split

# Load the generated dataset
df = pd.read_csv("synthetic_typos_title_er0.15.csv",comment="#")
tokenizer = DistilBertTokenizerFast.from_pretrained('distilbert-base-uncased')

def prepare_bigru_data(df, max_length=64):
    inputs, attention_masks, labels = [], [], []
    
    for _, row in df.iterrows():
        x_text = str(row['X_perturbed'])
        y_text = str(row['Y_target'])
        
        # Tokenize both X and Y normally (no need for regex pre-splitting!)
        x_encoded = tokenizer(x_text, truncation=True, padding='max_length', max_length=max_length)
        y_encoded = tokenizer(y_text, truncation=True, padding='max_length', max_length=max_length)
        
        x_ids = x_encoded['input_ids']
        y_ids = y_encoded['input_ids']
        
        # Initialize an all-zero label array
        subword_labels = [0.0] * max_length
        
        # Compare the token ID arrays to find exactly what changed
        sm = difflib.SequenceMatcher(None, y_ids, x_ids)
        
        for tag, i1, i2, j1, j2 in sm.get_opcodes():
            # If the sequences are not identical...
            if tag != 'equal':
                # j1 to j2 represents the indices of the corrupted subwords in X
                for j in range(j1, j2):
                    # Flag as 1.0 (Error), ensuring we don't flag [PAD], [CLS], or [SEP]
                    if x_ids[j] not in [tokenizer.cls_token_id, tokenizer.sep_token_id, tokenizer.pad_token_id]:
                        subword_labels[j] = 1.0
                        
        inputs.append(torch.tensor(x_ids))
        attention_masks.append(torch.tensor(x_encoded['attention_mask']))
        labels.append(torch.tensor(subword_labels))
        
    return torch.stack(inputs), torch.stack(attention_masks), torch.stack(labels)

print("Aligning masks to subwords using SequenceMatcher...")
X_ids, X_masks, y_labels = prepare_bigru_data(df)

# 80/20 Split
train_ids, val_ids, train_masks, val_masks, train_labels, val_labels = train_test_split(
    X_ids, X_masks, y_labels, test_size=0.2, random_state=42
)

class DetectionDataset(Dataset):
    def __init__(self, inputs, masks, labels):
        self.inputs, self.masks, self.labels = inputs, masks, labels
    def __len__(self): return len(self.inputs)
    def __getitem__(self, idx): return self.inputs[idx], self.masks[idx], self.labels[idx]

train_loader = DataLoader(DetectionDataset(train_ids, train_masks, train_labels), batch_size=32, shuffle=True)
val_loader = DataLoader(DetectionDataset(val_ids, val_masks, val_labels), batch_size=32)

print("Data loaded successfully! Ready for Bi-GRU training.")

Aligning masks to subwords using SequenceMatcher...
Data loaded successfully! Ready for Bi-GRU training.


In [3]:
import torch.nn as nn
from transformers import DistilBertModel

class BiGRUDetectionNetwork(nn.Module):
    def __init__(self, model_name='distilbert-base-uncased', hidden_dim=256):
        super().__init__()
        # Load DistilBERT strictly for its embedding layer
        distilbert = DistilBertModel.from_pretrained(model_name)
        self.embeddings = distilbert.embeddings
        
        # Freeze embeddings so we only train the Bi-GRU weights
        for param in self.embeddings.parameters():
            param.requires_grad = False
            
        embed_dim = distilbert.config.dim
        
        # The Bi-GRU layer
        self.bigru = nn.GRU(
            input_size=embed_dim, 
            hidden_size=hidden_dim, 
            num_layers=1, 
            bidirectional=True, 
            batch_first=True
        )
        
        # Project hidden states to a single binary logit
        self.classifier = nn.Linear(hidden_dim * 2, 1)

    def forward(self, input_ids, attention_mask):
        embeds = self.embeddings(input_ids)
        gru_out, _ = self.bigru(embeds)
        logits = self.classifier(gru_out).squeeze(-1) # Output shape: (batch_size, seq_len)
        return logits

/opt/amdgpu/share/libdrm/amdgpu.ids: No such file or directory


In [4]:
from torch.optim import AdamW
from tqdm import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = BiGRUDetectionNetwork().to(device)

print(device)

# reduction='none' allows us to apply the attention mask before calculating the mean
criterion = nn.BCEWithLogitsLoss(reduction='none')
optimizer = AdamW(model.parameters(), lr=1e-3)

epochs = 5

for epoch in range(epochs):
    # --- Training ---
    model.train()
    total_train_loss = 0
    
    loop = tqdm(train_loader, desc=f'Epoch {epoch+1}/{epochs} [Train]')
    for batch_ids, batch_masks, batch_labels in loop:
        batch_ids = batch_ids.to(device)
        batch_masks = batch_masks.to(device)
        batch_labels = batch_labels.to(device)
        
        optimizer.zero_grad()
        logits = model(batch_ids, batch_masks)
        
        # Raw loss per token
        loss = criterion(logits, batch_labels)
        
        # Mask out the padding tokens
        masked_loss = loss * batch_masks
        
        # Average loss over the ACTUAL number of tokens
        final_loss = masked_loss.sum() / batch_masks.sum()
        
        final_loss.backward()
        optimizer.step()
        
        total_train_loss += final_loss.item()
        loop.set_postfix(loss=final_loss.item())

    # --- Validation ---
    model.eval()
    total_val_loss = 0
    with torch.no_grad():
        for batch_ids, batch_masks, batch_labels in val_loader:
            batch_ids, batch_masks, batch_labels = batch_ids.to(device), batch_masks.to(device), batch_labels.to(device)
            logits = model(batch_ids, batch_masks)
            
            loss = criterion(logits, batch_labels)
            masked_loss = loss * batch_masks
            final_loss = masked_loss.sum() / batch_masks.sum()
            total_val_loss += final_loss.item()
            
    print(f"Epoch {epoch+1} Completed | Train Loss: {total_train_loss/len(train_loader):.4f} | Val Loss: {total_val_loss/len(val_loader):.4f}\n")

# Save the trained Bi-GRU weights
torch.save(model.state_dict(), "bigru_detection_model.pt")
print("Model saved successfully!")

Loading weights: 100%|██████████| 100/100 [00:00<00:00, 2121.99it/s]
DistilBertModel LOAD REPORT from: distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


cuda


Epoch 1/5 [Train]: 100%|██████████| 1461/1461 [00:32<00:00, 44.89it/s, loss=0.15]  


Epoch 1 Completed | Train Loss: 0.1847 | Val Loss: 0.1553



Epoch 2/5 [Train]: 100%|██████████| 1461/1461 [00:24<00:00, 58.50it/s, loss=0.0796]


...

Epoch 3 Completed | Train Loss: 0.0937 | Val Loss: 0.1468



Epoch 4/5 [Train]: 100%|██████████| 1461/1461 [00:24<00:00, 58.47it/s, loss=0.0232]


Epoch 4 Completed | Train Loss: 0.0650 | Val Loss: 0.1640



Epoch 5/5 [Train]: 100%|██████████| 1461/1461 [00:24<00:00, 59.76it/s, loss=0.0209]


Epoch 5 Completed | Train Loss: 0.0477 | Val Loss: 0.1771

Model saved successfully!


In [20]:
import torch

model.eval()

# Let's test one casual sentence and one "News" sentence
test_sentences = [
    "It is book today", 
    "Police use taser on python to tree man"
]

for text in test_sentences:
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    input_ids = inputs["input_ids"]
    
    with torch.no_grad():
        logits = model(input_ids, inputs["attention_mask"])
        # ONLY SIGMOID. No Softmax!
        probs = torch.sigmoid(logits).squeeze() 
        
    tokens = tokenizer.convert_ids_to_tokens(input_ids.squeeze())
    
    print(f"\nTesting: '{text}'")
    print("-" * 30)
    for token, prob in zip(tokens, probs):
        print(f"{token:>10} : {prob.item() * 100:>5.1f}% error risk")


Testing: 'It is book today'
------------------------------
     [CLS] :   0.0% error risk
        it :  21.5% error risk
        is :   0.8% error risk
      book :   5.6% error risk
     today :   1.4% error risk
     [SEP] :   0.1% error risk

Testing: 'Police use taser on python to tree man'
------------------------------
     [CLS] :   0.0% error risk
    police :   0.0% error risk
       use :   0.3% error risk
        ta :  41.3% error risk
     ##ser :  33.4% error risk
        on :   0.1% error risk
    python :  29.4% error risk
        to :   0.6% error risk
      tree :  88.4% error risk
       man :   0.8% error risk
     [SEP] :   0.0% error risk
